In [1]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=12"

import sys
sys.path.append('../../src/')

import numpy as onp
import jax
import jax.numpy as jnp

from myutils_corrected_2233 import Ntime, ACFs, set_detectors, set_detector_locations
from myutils_corrected_2233 import ln_likelihood_full_jit, Ntime, ACFs

from scipy.linalg import toeplitz
import scipy.signal as sig

import lal
from gwpy.timeseries import TimeSeries
from other_utils import bandpass_ds, analysis_data, interp1d_jax, load_tables

jax.config.update("jax_enable_x64", True) 

/Users/kallol/Work/Misc/ringdown__2/examples/full-sky/../../src/myutils_corrected_2233.py:3: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [2]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from corner import corner
from chainconsumer import Chain, ChainConsumer, Truth, ChainConfig, PlotConfig

import numpyro
from numpyro.contrib.nested_sampling import NestedSampler
import numpyro.distributions as dist
numpyro.enable_x64()

/Users/kallol/miniconda3/envs/skyloc_ringdown/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tgps = 1242442967.445 + 0.006 
seglen = 8 
fs = 16384 
fmin = 8 
event_id = "GW190521" 
plot_checks = 0 
T = 0.2 
srate = 4096 

ra = 3.5 
dec = 0.73 


factor = 10
seed       = 31567
t0         = 0.0
fmax       = 1500

qnm1_path  = "../../data/l2/n1l2m2.dat"
qnm2_path  = "../../data/l3/n1l3m3.dat"

tM_shifted_samples_save_dir = "./GW190521-t-shift-posteriors/"

In [4]:
dH1 = TimeSeries.fetch_open_data('H1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)
dL1 = TimeSeries.fetch_open_data('L1', tgps - seglen/2, tgps + seglen/2, sample_rate=fs)


n_analyze = Ntime(srate, T)

delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('H1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLHODIFF].location, ra, dec, tgps))
tH1 = tgps + dt_ifo

delays = {}
tgps_ = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps_)
dt_ifo = delays.get('L1',
                    lal.TimeDelayFromEarthCenter(lal.CachedDetectors[lal.LALDetectorIndexLLODIFF].location, ra, dec, tgps))
tL1 = tgps + dt_ifo

In [5]:
tH1, tL1

(1242442967.4318974, 1242442967.4302728)

In [6]:
dH1_cond = bandpass_ds(dH1, t0=tH1, ds=int(fs/srate), trim=0.25, f_min=fmin)
dL1_cond = bandpass_ds(dL1, t0=tH1, ds=int(fs/srate), trim=0.25, f_min=fmin)

In [7]:
nperseg = int(seglen * (1/(dH1.times.value[1] - dH1.times.value[0])))
noverlap = nperseg // 2

psdH = sig.welch(
    dH1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density",    
)

psdL = sig.welch(
    dL1.value,
    fs=fs,
    window="hann",
    nperseg=4096*4,
    noverlap=4096*2,
    detrend=False,
    return_onesided=True,
    scaling="density",   
)

In [8]:
dt_np = 1.0 / srate
Nt     = Ntime(srate=srate, T=T)
freqs_np = onp.fft.rfftfreq(Nt, d=dt_np)

fmin_eff = freqs_np[0]  if (fmin is None) else float(fmin)
fmax_eff = freqs_np[-1] if (fmax is None) else float(fmax)

i0 = int(onp.searchsorted(freqs_np, fmin_eff, side="left"))
i1 = int(onp.searchsorted(freqs_np, fmax_eff, side="right"))

freqs = jnp.asarray(freqs_np, dtype=jnp.float64)

# Prior limits
limits = [
    [30.0, 500.0],
    [0., 0.99],
    [0., 4.0],
    [0., 6.283185307179586],
    [0., 4.0],
    [0., 6.283185307179586],
    [-1., 1.],
    [0, 6.28],
    [-1., 1.],
    [0, 3.14]]

low  = onp.asarray([pair[0] for pair in limits], dtype=onp.float64)
high = onp.asarray([pair[1] for pair in limits], dtype=onp.float64)

In [9]:
psdH = interp1d_jax(jnp.asarray(psdH[0],dtype=jnp.float64), jnp.asarray(psdH[1],dtype=jnp.float64))
psdL = interp1d_jax(jnp.asarray(psdL[0],dtype=jnp.float64), jnp.asarray(psdL[1],dtype=jnp.float64))

omega_22_r, omega_22_i, omega_33_r, omega_33_i = load_tables(qnm1_path, qnm2_path)
tgps = lal.LIGOTimeGPS(tgps)
gmst = lal.GreenwichMeanSiderealTime(tgps)

rhoH, rhoL = ACFs(srate=srate, T=T, psdH=psdH, psdL=psdL, factor=factor)
covH=toeplitz(rhoH)
covL=toeplitz(rhoL)
L_H=jnp.linalg.cholesky(covH)
L_L=jnp.linalg.cholesky(covL)

resp_H_py = lal.CachedDetectors[lal.LALDetectorIndexLHODIFF].response
resp_L_py = lal.CachedDetectors[lal.LALDetectorIndexLLODIFF].response
resp_H = jnp.asarray(resp_H_py, dtype=jnp.float64)
resp_L = jnp.asarray(resp_L_py, dtype=jnp.float64)

set_detectors(resp_H, resp_L)

H1 = lal.CachedDetectors[lal.LALDetectorIndexLHODIFF]
L1 = lal.CachedDetectors[lal.LALDetectorIndexLLODIFF]

rH = jnp.array([H1.location[0], H1.location[1], H1.location[2]], dtype=jnp.float64)
rL = jnp.array([L1.location[0], L1.location[1], L1.location[2]], dtype=jnp.float64)

set_detector_locations(rH, rL)

In [10]:
MTSUN = lal.MTSUN_SI
tM = MTSUN * 325 # Remnant mass peak

In [11]:
num_shifts = 15

t_shifts = []
H1_data_t_shifted = []
L1_data_t_shifted = []

for jj in range(num_shifts):
    dH1_analysis_data = analysis_data(dH1_cond[1], dH1_cond[0], tH1 + jj*1*tM, n_analyze)
    dL1_analysis_data = analysis_data(dL1_cond[1], dL1_cond[0], tH1 + jj*1*tM, n_analyze)

    t_shifts.append(jj*1*tM)
    H1_data_t_shifted.append(dH1_analysis_data[1])
    L1_data_t_shifted.append(dL1_analysis_data[1])

In [12]:
onp.savetxt(tM_shifted_samples_save_dir + 't_shifts.txt', t_shifts)

In [13]:
low  = jnp.asarray(low,  dtype=jnp.float64)
high = jnp.asarray(high, dtype=jnp.float64)

In [14]:
logZs = []
for i in range(num_shifts):
    h_H_t = H1_data_t_shifted[i]
    h_L_t = L1_data_t_shifted[i]

    def make_loglik_fn(dataH, dataL, gmst, L_H, L_L,
                    omega_22_r, omega_22_i, omega_33_r, omega_33_i,
                    T, srate, t0):
        def _loglik(theta):
            return ln_likelihood_full_jit(
                dataH=dataH, dataL=dataL, params=theta,
                gmst=gmst, L_H=L_H, L_L=L_L,
                omega_22_r=omega_22_r, omega_22_i=omega_22_i,
                omega_33_r=omega_33_r, omega_33_i=omega_33_i,
                T=T, srate=srate, t0=t0
            )
        return _loglik

    loglik_fn = make_loglik_fn(
        dataH=h_H_t, dataL=h_L_t,
        gmst=gmst, L_H=L_H, L_L=L_L,
        omega_22_r=omega_22_r, omega_22_i=omega_22_i,
        omega_33_r=omega_33_r, omega_33_i=omega_33_i,
        T=T, srate=srate, t0=t0
    )

    def model():
        theta = numpyro.sample("theta", dist.Uniform(low, high).to_event(1))
        numpyro.factor("loglike", loglik_fn(theta))

    rng_key = jax.random.PRNGKey(int(seed) ^ 0xABCDEF)
    rng_run, rng_post = jax.random.split(rng_key)
    ns = NestedSampler(
        model,
        constructor_kwargs=dict(
            num_live_points=10000,    
            max_samples=500_000,
            verbose=True,
        ),
        termination_kwargs=dict(
            dlogZ=0.01,
        ),
    )
    ns.run(rng_run)
    ns.print_summary()
    posterior = ns.get_samples(rng_post, num_samples=100_000)

    onp.save(tM_shifted_samples_save_dir + 'posterior.' + event_id + '.' + str(i) + '.npy', onp.asarray(posterior['theta']))
    logZs.append(ns._results.log_Z_mean)

INFO:jaxns:Number of Markov-chains set to: 10000


Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondition(ess=None, evidence_uncert=None, live_evidence_frac=None, dlogZ=Array(0.01, dtype=float64, weak_type=True), max_samples=Array(500400, dtype=int64), max_num_likelihood_evaluations=None, log_L_contour=None, efficiency_threshold=None, rtol=None, atol=None, peak_XL_frac=None)
-------
Num samples: 5004
Num likelihood evals: 326565
Efficiency: 0.02973032386090201
log(L) contour: -2612.0625992474606
log(Z) est.: -832.7739687696787 +- 0.6608747027903127
log(Z | remaining) est.: 1788.2774583357675 +- 0.8373380768332809
ESS: 0.9128628908827155

-------
Num samples: 10008
Num likelihood evals: 694909
Efficiency: 0.01875087824482187
log(L) contour: -1287.2654523432382
log(Z) est.: -832.8286787419295 +- 0.524213504974915
log(Z | remaining) est.: 463.0084278182626 +- 0.6121303712321312
ESS: 1.5809420085871957

-

INFO:jaxns:Number of Markov-chains set to: 10000


Running over 12 devices.
Creating initial state with 10008 live points.
Running uniform sampling down to efficiency threshold of 0.1.
Running until termination condition: TerminationCondition(ess=None, evidence_uncert=None, live_evidence_frac=None, dlogZ=Array(0.01, dtype=float64, weak_type=True), max_samples=Array(500400, dtype=int64), max_num_likelihood_evaluations=None, log_L_contour=None, efficiency_threshold=None, rtol=None, atol=None, peak_XL_frac=None)
-------
Num samples: 5004
Num likelihood evals: 326573
Efficiency: 0.02973067713934324
log(L) contour: -2600.7931947186653
log(Z) est.: -836.1352401255322 +- 0.6530087815140599
log(Z | remaining) est.: 1773.162528525174 +- 0.7802543855320909
ESS: 0.940177179127701

-------
Num samples: 10008
Num likelihood evals: 693062
Efficiency: 0.018861097290696096
log(L) contour: -1294.3215308522833
log(Z) est.: -825.4678840071699 +- 0.7108011318576247
log(Z | remaining) est.: 477.0111089820175 +- 0.7512632966311601
ESS: 0.760594687295325

--

INFO:jaxns:Number of Markov-chains set to: 10000


...

--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 23561625
samples: 160128
phantom samples: 0
likelihood evals / sample: 147.1
phantom fraction (%): 0.0%
--------
logZ=-821.553 +- 0.036
max(logL)=-807.755
H=-9.02
ESS=25180
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 328.0 +- 81.0 | 235.0 / 309.0 / 458.0 | 334.0 | 334.0
theta[1]: 0.38 +- 0.24 | 0.07 / 0.35 / 0.73 | 0.87 | 0.87
theta[2]: 0.78 +- 0.68 | 0.2 / 0.55 / 1.71 | 1.31 | 1.31
theta[3]: 2.9 +- 1.8 | 0.5 / 2.9 / 5.5 | 0.6 | 0.6
theta[4]: 0.3 +- 0.36 | 0.02 / 0.17 / 0.72 | 0.46 | 0.46
theta[5]: 3.2 +- 1.8 | 0.7 / 3.1 / 5.7 | 2.8 | 2.8
theta[6]: -0.03 +- 0.38 | -0.53 / -0.02 / 0.47 | -0.46 | -0.46
theta[7]: 3.7 +- 1.6 | 1.6 / 3.4 / 5.7 | 2.7 | 2.7
theta[8]: -0.06 +- 0.6 | -0.83 / -0.15 / 0.78 | -0.72 | -0.72
theta[9]: 1.5 +- 0.91 | 0.31 / 1.46 / 2.79 | 2.3 | 2.3
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Runn

INFO:jaxns:Number of Markov-chains set to: 10000


--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 24593423
samples: 160128
phantom samples: 0
likelihood evals / sample: 153.6
phantom fraction (%): 0.0%
--------
logZ=-820.118 +- 0.037
max(logL)=-806.488
H=-9.27
ESS=23670
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 353.0 +- 92.0 | 237.0 / 341.0 / 475.0 | 322.0 | 322.0
theta[1]: 0.32 +- 0.23 | 0.05 / 0.26 / 0.67 | 0.88 | 0.88
theta[2]: 0.92 +- 0.77 | 0.23 / 0.67 / 2.02 | 0.36 | 0.36
theta[3]: 3.2 +- 1.8 | 0.6 / 3.3 / 5.6 | 1.5 | 1.5
theta[4]: 0.41 +- 0.46 | 0.04 / 0.26 / 0.98 | 0.13 | 0.13
theta[5]: 3.0 +- 1.8 | 0.5 / 3.0 / 5.6 | 3.7 | 3.7
theta[6]: -0.02 +- 0.4 | -0.54 / -0.02 / 0.51 | -0.22 | -0.22
theta[7]: 3.6 +- 1.6 | 1.4 / 3.3 / 5.7 | 4.8 | 4.8
theta[8]: -0.07 +- 0.59 | -0.84 / -0.14 / 0.76 | 0.63 | 0.63
theta[9]: 1.57 +- 0.9 | 0.33 / 1.6 / 2.8 | 3.08 | 3.08
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Runn

INFO:jaxns:Number of Markov-chains set to: 10000


--------
Termination Conditions:
Small remaining evidence
--------
likelihood evals: 27651281
samples: 180144
phantom samples: 0
likelihood evals / sample: 153.5
phantom fraction (%): 0.0%
--------
logZ=-827.494 +- 0.035
max(logL)=-810.907
H=-8.56
ESS=25273
--------
theta[#]: mean +- std.dev. | 10%ile / 50%ile / 90%ile | MAP est. | max(L) est.
theta[0]: 334.0 +- 90.0 | 231.0 / 315.0 / 468.0 | 383.0 | 383.0
theta[1]: 0.37 +- 0.25 | 0.06 / 0.32 / 0.76 | 0.98 | 0.98
theta[2]: 0.77 +- 0.71 | 0.17 / 0.53 / 1.69 | 0.28 | 0.28
theta[3]: 3.2 +- 1.8 | 0.6 / 3.2 / 5.6 | 6.1 | 6.1
theta[4]: 0.33 +- 0.38 | 0.03 / 0.2 / 0.78 | 0.18 | 0.18
theta[5]: 3.1 +- 1.8 | 0.7 / 3.1 / 5.6 | 1.6 | 1.6
theta[6]: -0.03 +- 0.4 | -0.56 / -0.02 / 0.49 | -0.26 | -0.26
theta[7]: 3.8 +- 1.6 | 1.5 / 4.2 / 5.8 | 4.4 | 4.4
theta[8]: -0.07 +- 0.61 | -0.84 / -0.17 / 0.78 | -0.49 | -0.49
theta[9]: 1.53 +- 0.9 | 0.31 / 1.52 / 2.8 | 2.78 | 2.78
--------
Running over 12 devices.
Creating initial state with 10008 live points.
Ru